## Nota de vigencia

Este notebook audita un modelo entrenado con el contrato de variables anterior. Sus resultados se conservan como histórico; hay que reentrenar y reejecutar la auditoría antes de compararlo con el cubo final.

# Auditoría completa de XGBoost

Entrenamiento 2019–2021, validación 2022 y test 2023 bloqueado. Las métricas iniciales se calculan sobre una muestra enriquecida de negativos: sirven para comparar, no son probabilidades poblacionales calibradas.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score,average_precision_score,roc_curve,precision_recall_curve,confusion_matrix,ConfusionMatrixDisplay,brier_score_loss
from src.modeling.data import TRAIN_YEARS,VALIDATION_YEARS,load_dataset_contract,sample_years_for_training
from src.modeling.features import resolve_feature_set
from src.modeling.evaluation import evaluate_binary_predictions
from xgboost import XGBClassifier
root=Path.cwd().resolve(); root=root if (root/'src').exists() else root.parent
contract=load_dataset_contract(root/'data/processed/tabular/egif'); features=resolve_feature_set(contract.predictors,'temporal_compacto')
train=sample_years_for_training(contract,TRAIN_YEARS,contract.predictors,25,0); val=sample_years_for_training(contract,VALIDATION_YEARS,contract.predictors,25,0)
ratio=(train.target_ignicion==0).sum()/(train.target_ignicion==1).sum(); model=XGBClassifier(n_estimators=500,max_depth=6,learning_rate=.05,subsample=.8,colsample_bytree=.8,scale_pos_weight=ratio,eval_metric='aucpr',n_jobs=-1,random_state=42).fit(train[features],train.target_ignicion)
p=model.predict_proba(val[features])[:,1]; y=val.target_ignicion.to_numpy(); print({'features':len(features),'train_rows':len(train),'val_rows':len(val),'val_positives':int(y.sum()),'roc_auc':roc_auc_score(y,p),'pr_auc':average_precision_score(y,p),'brier_sample':brier_score_loss(y,p),**evaluate_binary_predictions(y,p)})

In [ ]:
fpr,tpr,_=roc_curve(y,p); precision,recall,_=precision_recall_curve(y,p)
fig,ax=plt.subplots(1,2,figsize=(12,4)); ax[0].plot(fpr,tpr); ax[0].plot([0,1],[0,1],'--'); ax[0].set(title='ROC',xlabel='FPR',ylabel='TPR'); ax[1].plot(recall,precision); ax[1].set(title='Precision-Recall',xlabel='Recall',ylabel='Precision'); plt.show()
for fraction in (.01,.05,.10):
 threshold=np.quantile(p,1-fraction); print(f'Top {fraction:.0%}:',int(((p>=threshold)&(y==1)).sum()),'/',int(y.sum()),'igniciones')
threshold=np.quantile(p,.99); ConfusionMatrixDisplay(confusion_matrix(y,p>=threshold)).plot(); plt.title('Matriz de confusión: top 1% de riesgo')

In [ ]:
importance=pd.Series(model.feature_importances_,index=features).sort_values(ascending=False); display(importance.head(25).to_frame('gain')); importance.head(25).sort_values().plot.barh(figsize=(8,9),title='Importancia XGBoost (gain)')
# SHAP es opcional: se limita a 2.000 filas para mantener memoria controlada.
try:
 import shap
 sample=val[features].sample(min(2000,len(val)),random_state=42); shap.summary_plot(shap.TreeExplainer(model).shap_values(sample),sample,show=True)
except ImportError: print('SHAP no instalado; la importancia por gain sigue disponible.')

In [ ]:
val['month']=pd.to_datetime(val.fecha).dt.month; monthly=[]
for month,frame in val.groupby('month'):
 if frame.target_ignicion.nunique()==2: monthly.append({'month':month,'n':len(frame),'positives':int(frame.target_ignicion.sum()),**evaluate_binary_predictions(frame.target_ignicion,model.predict_proba(frame[features])[:,1])})
monthly=pd.DataFrame(monthly); display(monthly); monthly.plot(x='month',y=['pr_auc','recall_at_top_fraction'],marker='o',title='Estabilidad mensual 2022')